In [3]:
# combining Costs file with Markets

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [5]:
df = pd.read_csv("output/measure_costs_benefits.csv")
df

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,TRC_BCR,SCT_cost,SCT_benefit,SCT_BCR,RIM_cost,RIM_benefit,RIM_BCR,PCT_cost,PCT_benefit,PCT_BCR
0,refrigerator_electricity_efficient_residential...,residential,LIRRet,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,4.143571,1430.0,11274.677814,7.884390,2584.500000,6336.406010,2.451695,650,8560.099876,13.169384
1,refrigerator_electricity_efficient_residential...,residential,NLIRRet,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,4.304174,1430.0,11569.585903,8.090620,3592.500001,6566.068557,1.827716,650,9797.762424,15.073481
2,refrigerator_electricity_efficient_residential...,residential,LIRRet,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,4.304174,1430.0,11569.585903,8.090620,3592.500001,6566.068557,1.827716,650,9797.762424,15.073481
3,refrigerator_electricity_efficient_residential...,residential,NLIRRet,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,4.143571,1430.0,11277.245611,7.886186,2584.500000,6336.406010,2.451695,650,8560.099876,13.169384
4,refrigerator_electricity_efficient_residential...,residential,LIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,5.856841,990.0,11114.342065,11.226608,1714.500000,6053.372503,3.530693,450,7547.066369,16.771259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,74.218172,184.8,NaN,NaN,84.700000,13536.119510,159.812509,77,13787.058014,179.052701
148,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,74.218172,184.8,NaN,NaN,84.700000,13536.119510,159.812509,77,13787.058014,179.052701
149,HERS_electricity_efficient_residential_whole_h...,residential,NLIRNC,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-2.795412,-4920.0,NaN,NaN,-1550.000000,11914.969669,-7.687077,-2050,12445.508173,-6.070980
150,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-2.795412,-4920.0,NaN,NaN,-1550.000000,11914.969669,-7.687077,-2050,12445.508173,-6.070980


In [6]:
# make a list from df which is efficient_condition and measure_life_(yrs) and drop duplicates

#this will have to come from the workpapers 
df_eff_life = df[['efficient_condition', 'measure_life_(yrs)']].drop_duplicates()
df_base_life = df[['baseline_condition', 'measure_life_(yrs)']].drop_duplicates()
# now just stack these two dataframes on top of each other with the columns efficient_condition and baseline_condition renamed to condition
df_eff_life = df_eff_life.rename(columns={'efficient_condition': 'condition', 'measure_life_(yrs)': 'measure_life_(yrs)'})
df_base_life = df_base_life.rename(columns={'baseline_condition': 'condition', 'measure_life_(yrs)': 'measure_life_(yrs)'})

df_life = pd.concat([df_eff_life, df_base_life]).drop_duplicates().reset_index(drop=True)
df_life

,condition,measure_life_(yrs)
0,refrigerator_electricity_efficient_residential,10
1,refrigerator_electricity_top10_residential,10
2,insulation_natural_gas_efficient_residential,30
3,HERS_electricity_efficient_residential,30
4,refrigerator_electricity_existing_residential,10
5,refrigerator_electricity_baseline_residential,10
6,whole_home_natural_gas_baseline_residential,30


In [7]:
df_yr1 = pd.read_pickle("df_yr1.pkl")
# first need to change df_yr1 into long format with the current column names which are building types into rows and a new column for building type
df_yr1 = df_yr1.melt(id_vars=['condition_name', 'competition_group', 'subgroup', 'electric_utility', 'gas_utility'], var_name='building_type', value_name='intial_count')
df_yr1
# now I am going to allocate the initial counts into the markets based on the stock and flow proportions from expert opinion
# there are three markets that I want to allocate into: RET_ER, REMAINING, and ROB 
# ROB is 1/EUL (measure_life (yrs)
# RET_ER is 1/3 of non ROB stock
# remaining is 2/3 of non ROB stock

# use df_life to map measure life to df_yr1 based on efficient_condition
df_yr1 = pd.merge(df_yr1, df_life, left_on='condition_name', right_on='condition', how='left')

df_yr1['ROB'] = df_yr1['intial_count'] / df_yr1['measure_life_(yrs)']  # assuming measure life of 15 years
df_yr1['RET_ER'] = (df_yr1['intial_count'] - df_yr1['ROB']) * (1/3)
df_yr1['REMAINING'] = (df_yr1['intial_count'] - df_yr1['ROB']) * (2/3)

df_yr1 = df_yr1.melt(id_vars=['condition_name', 'competition_group', 'subgroup', 'electric_utility', 'gas_utility', 'building_type', 'measure_life_(yrs)', 'intial_count', 'condition'], var_name='market', value_name='count')
df_yr1

#need to remove the text '_year_one' from the building_type column
df_yr1['building_type'] = df_yr1['building_type'].str.replace('_year_one', '')

In [8]:
# until assembled measures is complete we will just drop NAs
df_yr1 = df_yr1.dropna(subset=['measure_life_(yrs)'])
df_yr1

,condition_name,competition_group,subgroup,electric_utility,gas_utility,building_type,measure_life_(yrs),intial_count,condition,market,count
56,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,test_utility_1,single_family,10.0,12500.0,refrigerator_electricity_existing_residential,ROB,1250.0
57,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,test_utility_2,single_family,10.0,25000.0,refrigerator_electricity_existing_residential,ROB,2500.0
58,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_2,test_utility_2,single_family,10.0,68750.0,refrigerator_electricity_existing_residential,ROB,6875.0
59,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,none,single_family,10.0,12500.0,refrigerator_electricity_existing_residential,ROB,1250.0
60,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_2,none,single_family,10.0,13750.0,refrigerator_electricity_existing_residential,ROB,1375.0
...,...,...,...,...,...,...,...,...,...,...,...
1493,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_2,test_utility_2,multi_family_li,30.0,350.0,insulation_natural_gas_efficient_residential,REMAINING,225.555556
1494,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_1,none,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,80.555556
1495,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_2,none,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,80.555556
1496,insulation_natural_gas_efficient_residential,whole_home,none,none,test_utility_1,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,80.555556


In [9]:
# now we just merge df_yr1 and df on condition_name, competition_group, electric_utility, gas_utility, building_type, and market
df_yrs = pd.merge(df_yr1, df, left_on=['condition_name', 'electric_utility', 'gas_utility', 'building_type', 'market'], right_on=['efficient_condition',  'electric_utility', 'gas_utility', 'building_type', 'market'], how='left')
df_yrs


,condition_name,competition_group,subgroup,electric_utility,gas_utility,building_type,measure_life_(yrs)_x,intial_count,condition,market,...,TRC_BCR,SCT_cost,SCT_benefit,SCT_BCR,RIM_cost,RIM_benefit,RIM_BCR,PCT_cost,PCT_benefit,PCT_BCR
0,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,test_utility_1,single_family,10.0,12500.0,refrigerator_electricity_existing_residential,ROB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,test_utility_2,single_family,10.0,25000.0,refrigerator_electricity_existing_residential,ROB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_2,test_utility_2,single_family,10.0,68750.0,refrigerator_electricity_existing_residential,ROB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_1,none,single_family,10.0,12500.0,refrigerator_electricity_existing_residential,ROB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,refrigerator_electricity_existing_residential,refrigeration,full_size,test_utility_2,none,single_family,10.0,13750.0,refrigerator_electricity_existing_residential,ROB,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_2,test_utility_2,multi_family_li,30.0,350.0,insulation_natural_gas_efficient_residential,REMAINING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
416,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_1,none,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
417,insulation_natural_gas_efficient_residential,whole_home,none,test_utility_2,none,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
418,insulation_natural_gas_efficient_residential,whole_home,none,none,test_utility_1,multi_family_li,30.0,125.0,insulation_natural_gas_efficient_residential,REMAINING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
#Not removing NAs we need to make sure all the files have the shared column names and such

df_yrs.to_pickle("df_yrs.pkl")